# 03 · Diagnóstico operativo

Este notebook profundiza en las causas de retraso, la interacción clima–causa, hotspots ruta × hora y escenarios de impacto operativo.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
REPORTS = PROJECT_ROOT / 'reports'
FIGURES = REPORTS / 'figures'

for p in [DATA_PROCESSED, REPORTS, FIGURES]:
    p.mkdir(parents=True, exist_ok=True)

candidate_paths = [
    DATA_PROCESSED / 'ferry_operations_processed.csv',
    DATA_RAW / 'ferry_operations_raw.csv',
    PROJECT_ROOT / 'ferry_operations_raw.csv'
]
csv_path = next((p for p in candidate_paths if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError('No se encuentra el dataset. Ejecuta primero 01 y 02, o sube ferry_operations_raw.csv.')

df = pd.read_csv(csv_path)
print('Archivo cargado:', csv_path)
print('Shape:', df.shape)
df.head()


In [ ]:
# Preparación de columnas
for c in ['scheduled_start','actual_start','scheduled_end','actual_end','date']:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors='coerce')

for c in ['canceled','on_time','maintenance_flag']:
    if c in df.columns:
        df[c] = df[c].astype(str).str.lower().isin(['true','1','yes','si','sí'])

df['completed'] = ~df['canceled']
df['hour'] = df['scheduled_start'].dt.hour
df['month'] = df['scheduled_start'].dt.month
df['delay_min_clean'] = df['delay_min'].fillna(0)
df['on_time_15'] = (df['delay_min'] <= 15) & df['completed']
completed = df[df['completed']].copy()

print('Servicios completados:', len(completed))


In [ ]:
# Pareto de causas de retraso
pareto = (
    completed.groupby('disruption_reason')
      .agg(
          total_delay_min=('delay_min','sum'),
          avg_delay_min=('delay_min','mean'),
          p95_delay_min=('delay_min', lambda x: x.quantile(0.95)),
          services=('operation_id','count')
      )
      .sort_values('total_delay_min', ascending=False)
      .reset_index()
)
pareto['share_%'] = pareto['total_delay_min'] / pareto['total_delay_min'].sum() * 100
pareto['cum_share_%'] = pareto['share_%'].cumsum()
pareto.to_csv(REPORTS / 'pareto_delay_causes.csv', index=False)
pareto


In [ ]:
# Figura: Pareto
fig, ax1 = plt.subplots(figsize=(11,6))
ax1.bar(pareto['disruption_reason'], pareto['total_delay_min'])
ax1.set_ylabel('Retraso total acumulado (min)')
ax1.set_xlabel('Causa de disrupción')
ax1.tick_params(axis='x', rotation=35)

ax2 = ax1.twinx()
ax2.plot(pareto['disruption_reason'], pareto['cum_share_%'], marker='o')
ax2.set_ylabel('Share acumulado (%)')
ax2.set_ylim(0, 105)
plt.title('Pareto de causas por retraso total')
plt.tight_layout()
plt.savefig(FIGURES / 'pareto_delay.png', dpi=160)
plt.show()


In [ ]:
# Interacción clima × causa
weather_reason = (
    completed.groupby(['weather','disruption_reason'])
      .agg(
          services=('operation_id','count'),
          avg_delay_min=('delay_min','mean'),
          p95_delay_min=('delay_min', lambda x: x.quantile(0.95)),
          otr_15=('on_time_15','mean')
      )
      .reset_index()
)
weather_reason.to_csv(REPORTS / 'weather_reason_diagnostics.csv', index=False)
weather_reason.sort_values('avg_delay_min', ascending=False).head(15)


In [ ]:
# Figura: heatmap clima × causa
pivot_wr = weather_reason.pivot(index='weather', columns='disruption_reason', values='avg_delay_min')
plt.figure(figsize=(12,5))
sns.heatmap(pivot_wr, annot=True, fmt='.1f', cmap='viridis')
plt.title('Delay medio por clima y causa de disrupción')
plt.xlabel('Causa de disrupción')
plt.ylabel('Clima')
plt.tight_layout()
plt.savefig(FIGURES / 'heatmap_weather_reason_delay.png', dpi=160)
plt.show()


In [ ]:
# Figura: boxplot por clima
plt.figure(figsize=(10,5))
sns.boxplot(data=completed, x='weather', y='delay_min')
plt.title('Distribución de retrasos por condición meteorológica')
plt.xlabel('Clima')
plt.ylabel('Delay (min)')
plt.tight_layout()
plt.savefig(FIGURES / 'boxplot_delay_by_weather.png', dpi=160)
plt.show()


In [ ]:
# Hotspots ruta × hora
hotspots = (
    completed.groupby(['route_id','hour'])
      .agg(
          services=('operation_id','count'),
          avg_delay_min=('delay_min','mean'),
          p95_delay_min=('delay_min', lambda x: x.quantile(0.95)),
          otr_15=('on_time_15','mean'),
          avg_margin=('margin','mean')
      )
      .reset_index()
)

hotspots = hotspots[hotspots['services'] >= 5].sort_values('avg_delay_min', ascending=False)
hotspots.to_csv(REPORTS / 'route_hour_hotspots.csv', index=False)
hotspots.head(20)


In [ ]:
# Figura: heatmap ruta × hora
pivot_rh = hotspots.pivot(index='route_id', columns='hour', values='avg_delay_min')
plt.figure(figsize=(14,6))
sns.heatmap(pivot_rh, cmap='magma', linewidths=.2)
plt.title('Hotspots operativos: delay medio por ruta y hora')
plt.xlabel('Hora programada')
plt.ylabel('Ruta')
plt.tight_layout()
plt.savefig(FIGURES / 'heatmap_route_hour_delay.png', dpi=160)
plt.show()


In [ ]:
# Escenarios de impacto: reducir causas principales un 10%
total_delay = completed['delay_min'].sum()
impact_rows = []
for cause in ['WEATHER','TECHNICAL','PORT_CONGESTION','LATE_BOARDING']:
    cause_delay = completed.loc[completed['disruption_reason'].eq(cause), 'delay_min'].sum()
    saving = cause_delay * 0.10
    impact_rows.append({
        'scenario': f'Reducir {cause} 10%',
        'saved_minutes': saving,
        'saved_delay_share_%': saving / total_delay * 100
    })

combined = completed.loc[completed['disruption_reason'].isin(['WEATHER','TECHNICAL']), 'delay_min'].sum() * 0.10
impact_rows.append({
    'scenario': 'Reducir WEATHER 10% + TECHNICAL 10%',
    'saved_minutes': combined,
    'saved_delay_share_%': combined / total_delay * 100
})

impact = pd.DataFrame(impact_rows).sort_values('saved_minutes', ascending=False)
impact.to_csv(REPORTS / 'impact_scenarios.csv', index=False)
impact


In [ ]:
# Figura: escenarios de impacto
plt.figure(figsize=(10,5))
plt.barh(impact['scenario'], impact['saved_minutes'])
plt.gca().invert_yaxis()
plt.title('Escenarios de ahorro de retraso agregado')
plt.xlabel('Minutos de retraso ahorrados')
plt.tight_layout()
plt.savefig(FIGURES / 'impact_scenarios.png', dpi=160)
plt.show()


In [ ]:
# Texto de recomendaciones ejecutivas
top_causes = pareto.head(3)['disruption_reason'].tolist()
top_hotspots = hotspots.head(5)[['route_id','hour','avg_delay_min','otr_15']]

recommendations = f'''# Recomendaciones operativas

## 1. Plan de contingencia por meteorología
WEATHER aparece como una de las principales fuentes de retraso agregado. Se recomienda definir umbrales operativos para WINDY, ROUGH y STORM con buffers adicionales, comunicación preventiva y revisión de slots.

## 2. Fiabilidad técnica
TECHNICAL presenta alto retraso medio y se amplifica con mal clima. Se recomienda reforzar checks pre-salida y mantenimiento preventivo en rutas críticas.

## 3. Congestión portuaria
PORT_CONGESTION debe gestionarse con buffers, coordinación portuaria y revisión de ventanas críticas.

## 4. Hotspots operativos
Priorizar análisis semanal de rutas y horas con peor delay medio y bajo OTR≤15.

## Top causas
{', '.join(top_causes)}

## Top hotspots
{top_hotspots.to_markdown(index=False)}
'''

(REPORTS / 'operational_recommendations.md').write_text(recommendations, encoding='utf-8')
print(recommendations)
